# tokens

> Design tokens: the shared CSS custom properties for colour, space, radius, type, motion, and elevation

In [ ]:
#| default_exp tokens

Tokens are the shared values that components use for styling. Changing a token changes all the components that read it. We register each family of tokens with `reg.rule` in the `tokens` layer. An application can override any of them with ordinary CSS.

In [ ]:
from importlib.resources import files
from fastcore.utils import *
from answerui.registry import reg, Registry

In [ ]:
from fastcore.test import *
from fasthtml.common import Div
from answerui.demo import demo


## Colour

The palette notebook produced twelve CSS custom properties: neutral and vivid ladders derived from `--hue`, with light and dark values for each. Here we set the application's default `--hue` and declare named colour contexts.

In [ ]:
PALETTE = files('answerui').joinpath('css/palette.css').read_text()

In [ ]:
print(f'{(_ := PALETTE.splitlines())[0][:100]}…\n…\n{_[-1][:100]}…')

--n-3: light-dark(oklch(0.9914 clamp(0, 0.00801 + 0.00551*cos((1*var(--hue) - 136.63)*1deg) + 0.0032…
…
--v2: light-dark(oklch(0.4111 clamp(0, 0.12402 + 0.05381*cos((1*var(--hue) - 328.74)*1deg) + 0.02534…


In [ ]:
rungs = re.findall(r'^--([\w-]+):', PALETTE, re.M)
test_eq(len(rungs), 12)


### Application palette

We use hue 260 as the default. `color-scheme: light dark` lets the browser follow the user's preference.

In [ ]:
_hue_css = ':root { --hue: 260; color-scheme: light dark; }'

### Colour contexts

Each of the four severity contexts, `danger`, `warning`, `success`, and `info`, has its own hue and a bare attribute of the same name. These hues are independent of the application hue.

In [ ]:
CONTEXTS = dict(danger=25, warning=75, success=145, info=230)

A custom property is computed on the element that declares it. An element that sets only `--hue` inherits rungs already computed from its parent's hue. A colour context therefore needs both `--hue` and the ladders declared on its selector. `reg.context` records the selector and its hue. Whenever the stylesheet is assembled, the `palette` rule sets `--hue` on every recorded context and declares the ladders on `:root` and on each context. A later module or an application can therefore add a context without repeating the ladders:


In [ ]:
@patch(as_prop=True)
def contexts(self:Registry):
    "The colour contexts declared on this registry, as a dict from selector to hue"
    return self.__dict__.setdefault('_contexts', {})

@patch
def context(
    self:Registry,
    sel, # Selector of the context
    hue, # Its hue: a number, or an expression such as `var(--hue-danger)`
):
    "Declare `sel` as a colour context whose palette derives from `hue`"
    self.contexts[sel] = hue
    self.version += 1


In [ ]:
def _context_css(r): return '\n'.join(f'{sel} {{ --hue: {hue}; }}' for sel,hue in r.contexts.items())
def _palette_css(r): return f"{_context_css(r)}\n{', '.join([':root', *r.contexts])} {{\n{PALETTE}\n}}"


We declare the four severity contexts first:

In [ ]:
for name in CONTEXTS: reg.context(f'[{name}]', f'var(--hue-{name})')

In [ ]:
print(_context_css(reg))

[danger] { --hue: var(--hue-danger); }
[warning] { --hue: var(--hue-warning); }
[success] { --hue: var(--hue-success); }
[info] { --hue: var(--hue-info); }


We register the application default, context hues, and palette declarations in the `tokens` layer.

In [ ]:
_hues = ' '.join(f'--hue-{name}: {hue};' for name,hue in CONTEXTS.items())
reg.rule('hue', _hue_css, layer='tokens', doc='The application hue and the supported colour schemes.')
reg.rule('contexts', f':root {{ {_hues} }}', layer='tokens', doc='The severity hues, each independently adjustable.')
reg.rule('palette', partial(_palette_css, reg), layer='tokens', doc='The neutral and vivid ladders, declared on the root and on every colour context.')


palette [tokens]: The neutral and vivid ladders, declared on the root and on every colour context.

`swatches` shows tokens as a row of tiles. Each tile's inline style sets one CSS property to one named token:

In [ ]:
def swatches(prop, names, **kw):
    "A row of tiles, each with CSS property `prop` set to one of the named tokens"
    tiles = [Div(n, style=f'padding:.5rem;background:var(--n-2);font:12px monospace;{prop}:var(--{n})') for n in names]
    return Div(*tiles, style='display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)', **kw)

The ladders under the application hue, in a danger context, and in a region with a dark colour scheme:

In [ ]:
demo(swatches('background', rungs), swatches('background', rungs, danger=True),
    Div(swatches('background', rungs), style='color-scheme:dark'))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-3)">n-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-2)">n-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-1)">n-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n)">n</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n1)">n1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n2)">n2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-3)">v-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-2)">v-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-1)">v-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v)">v</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v1)">v1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v2)">v2</div>
</div>
<div danger style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-3)">n-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-2)">n-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-1)">n-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n)">n</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n1)">n1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n2)">n2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-3)">v-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-2)">v-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-1)">v-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v)">v</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v1)">v1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v2)">v2</div>
</div>
<div style="color-scheme:dark">
  <div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-3)">n-3</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-2)">n-2</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-1)">n-1</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n)">n</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n1)">n1</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n2)">n2</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-3)">v-3</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-2)">v-2</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-1)">v-1</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v)">v</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v1)">v1</div>
    <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v2)">v2</div>
  </div>
</div>

```

### Tint

A control drawn as plain text, with no background of its own, shows its hover state as a tint: a faint layer of its own text colour. One `--tint` token works for body text, the accent, and severity contexts, because it is derived from `currentColor`.

In [ ]:
reg.rule('tint', ':root { --tint: oklch(from currentColor l c h / .08); }', layer='tokens', doc='A faint layer of the current text colour, shown by a faceless control when hovered.')

tint [tokens]: A faint layer of the current text colour, shown by a faceless control when hovered.

In [ ]:
def tinted(label, style=''):
    "A tile tinted by its own text colour"
    return Div(label, style=f'padding:.5rem 1rem;background:var(--tint);border-radius:var(--radius-1);{style}')

demo(Div(tinted('Body text'), tinted('Accent', 'color:var(--v)'), Div(tinted('Danger', 'color:var(--v)'), danger=True),
    style='display:flex;gap:1rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)'))

```html
<div style="display:flex;gap:1rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem 1rem;background:var(--tint);border-radius:var(--radius-1);">Body text</div>
  <div style="padding:.5rem 1rem;background:var(--tint);border-radius:var(--radius-1);color:var(--v)">Accent</div>
  <div danger>
    <div style="padding:.5rem 1rem;background:var(--tint);border-radius:var(--radius-1);color:var(--v)">Danger</div>
  </div>
</div>

```

## Typography

There are five sizes: the body size, `--text`, and four derived from it. `--text` is declared on `:root`, and the four derived sizes on every element. Setting `--text` on a region therefore gives the region and everything inside it a scale of its own. Text inherits its `font-size` rather than reading `--text`. For its text to change size, the region also needs `font-size: var(--text)`. We also declare regular and semibold weights, plus normal and tight line heights.

The tokens style no element themselves. The base layer styles native elements, and its [typrose](https://github.com/AnswerDotAI/typrose) rule styles prose regions.

In [ ]:
_type_css = r"""
:root {
  --font: 'IBM Plex Sans', system-ui, sans-serif;
  --font-serif: 'IBM Plex Serif', Georgia, serif;
  --font-mono: 'IBM Plex Mono', ui-monospace, Menlo, Consolas, monospace;
  --text: 1rem;
  --weight: 400; --weight1: 600;
  --leading-1: 1.25; --leading: 1.5;
}
*, ::before, ::after {
  --text-1: calc(var(--text) * .875);
  --text1: calc(var(--text) * 1.25);
  --text2: calc(var(--text) * 1.5);
  --text3: calc(var(--text) * 2);
}
"""

In [ ]:
reg.rule('type', _type_css, layer='tokens',
    doc='The three font families, five sizes, regular and semibold weights, and two line heights.')

type [tokens]: The three font families, five sizes, regular and semibold weights, and two line heights.

In [ ]:
demo(swatches('font-size', ['text-1', 'text', 'text1', 'text2', 'text3']))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text-1)">text-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text)">text</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text1)">text1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text2)">text2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text3)">text3</div>
</div>

```

### Fonts

We use IBM Plex: Plex Sans for text and headings, and Plex Mono for code. Plex Serif is also included, for apps that want serif headings.

The font files are shipped in the package. When an app calls `reg.serve`, they are served from the app itself. Otherwise the browser fetches them from jsDelivr.


`font_face` registers one font file with `reg.file` and returns its `@font-face` rule. We register all five `@font-face` rules as one registry rule, `fonts`:


In [ ]:
FONTS = [('IBM Plex Sans', 400, 'normal'), ('IBM Plex Sans', 400, 'italic'), ('IBM Plex Sans', 600, 'normal'),
    ('IBM Plex Serif', 600, 'normal'), ('IBM Plex Mono', 400, 'normal')]

def font_face(
    family, # Family name, as CSS uses it
    weight, # Numeric weight of this file
    style,  # `normal` or `italic`
    version='5.3.0', # Version of the family's Fontsource package the public URL names
):
    "Register one packaged font file with `reg.file` and return its `@font-face` rule"
    slug = family.lower().replace(' ', '-')
    name = f'{slug}-latin-{weight}-{style}.woff2'
    url = reg.file(name, files('answerui').joinpath('fonts', name), f'https://cdn.jsdelivr.net/npm/@fontsource/{slug}@{version}/files/{name}')
    return f"@font-face {{ font-family: '{family}'; font-weight: {weight}; font-style: {style}; font-display: swap; src: url('{url}') format('woff2'); }}"

In [ ]:
reg.rule('fonts', '\n'.join(font_face(*f) for f in FONTS), layer='tokens', doc='IBM Plex Sans, Serif and Mono, from the package when served and from jsDelivr otherwise.')

fonts [tokens]: IBM Plex Sans, Serif and Mono, from the package when served and from jsDelivr otherwise.

Here are the three families. Nothing uses `--font-serif` unless an app asks for it:


In [ ]:
demo(*[Div('The quick brown fox jumps over the lazy dog, 0123456789', style=f'font-family: var(--{t})') for t in ('font', 'font-serif', 'font-mono')])


```html
<div style="font-family: var(--font)">The quick brown fox jumps over the lazy dog, 0123456789</div>
<div style="font-family: var(--font-serif)">The quick brown fox jumps over the lazy dog, 0123456789</div>
<div style="font-family: var(--font-mono)">The quick brown fox jumps over the lazy dog, 0123456789</div>

```

## Space

One spacing scale sets the padding inside components and the gaps between them. Every element derives the steps from `--space`. Changing `--space` on `:root` changes spacing across the application, and changing it on a region gives that region a scale of its own. Gaps are set by the layout parent, not by outer margins on the components.

Components read the steps derived from `--space`. Setting a step such as `--space1` directly on a region changes it for that element alone, because each descendant derives its own steps from the `--space` it inherits.

In [ ]:
reg.rule('space', r"""
:root { --space: 1rem; }
*, ::before, ::after {
  --space-3: calc(var(--space) * .25);
  --space-2: calc(var(--space) * .5);
  --space-1: calc(var(--space) * .75);
  --space1: calc(var(--space) * 1.5);
  --space2: calc(var(--space) * 2);
  --space3: calc(var(--space) * 3);
}
""", layer='tokens', doc='A spacing scale derived from one application-wide base value.')

space [tokens]: A spacing scale derived from one application-wide base value.

In [ ]:
demo(swatches('padding', ['space-3', 'space-2', 'space-1', 'space', 'space1', 'space2', 'space3']))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-3)">space-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-2)">space-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-1)">space-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space)">space</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space1)">space1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space2)">space2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space3)">space3</div>
</div>

```

## Radius

There are three corner radii: `--radius` and two derived from it on every element, as with space. Setting `--radius` on `:root` changes roundness across the application, and setting it on a region changes that region's roundness. `--radius-full` is a fixed 9999px for pill shapes and doesn't derive from `--radius`.

In [ ]:
reg.rule('radius', r"""
:root { --radius: .5rem; --radius-full: 9999px; }
*, ::before, ::after {
  --radius-1: calc(var(--radius) * .5);
  --radius1: calc(var(--radius) * 2);
}
""", layer='tokens', doc='Three corner radii derived from one base value, plus an independent pill radius.')

radius [tokens]: Three corner radii derived from one base value, plus an independent pill radius.

In [ ]:
demo(swatches('border-radius', ['radius-1', 'radius', 'radius1', 'radius-full']))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius-1)">radius-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius)">radius</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius1)">radius1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius-full)">radius-full</div>
</div>

```

## Motion

`--motion` sets the shared duration, and `--ease` the easing curve. The `aui-spin` keyframes that busy controls share are also defined here, for any component to use. Components use the duration directly or multiply it for longer effects. Temporarily increasing `--motion` lets us inspect transitions in slow motion.

Effects timed by `--motion` are instant when the user prefers reduced motion, because `--motion` is then `0ms`. Components must remain functional without motion and must not depend on an animation finishing to perform essential work.

In [ ]:
reg.rule('motion', r"""
:root {
  --motion: 150ms;
  --ease: cubic-bezier(.2, 0, 0, 1);
}
@media (prefers-reduced-motion: reduce) {
  :root { --motion: 0ms; }
}
@keyframes aui-spin { to { rotate: 1turn; } }
""", layer='tokens', doc='A shared duration and easing curve, stilled under reduced motion, and the spin shared by busy controls.')

motion [tokens]: A shared duration and easing curve, stilled under reduced motion, and the spin shared by busy controls.

## Elevation

answerui shows depth by relief: each surface is drawn as if lit by one light from the top left. A raised surface, such as a button, has a light rim along its top and left edges, a dark rim along its bottom and right edges, and a small shadow beneath it. A well, such as a text input, has the same lighting the other way round: dark rim at the top and left, light rim at the bottom and right, and a shadow inside its top edge. With one light for every surface, a reader can tell which surfaces stand off the page and which are cut into it.

Two values control the effect: `--face` and `--relief`. `--face` is the colour of the surface. The rim colours are derived from it: `--hi` is a little lighter than the face, and `--lo` a little darker. They are not taken from a rung of the ladder, because a rung that is lighter than the face in light mode is darker than it in dark mode. A light rim must stay lighter than its face in both modes, and a dark rim darker. `--relief` scales the opacity of the rims and shadows: `1` is full strength, `.5` half, and `0` none. A design with `--relief: 0` is flat, with every face, border, and state colour unchanged. `--relief` also scales how far a button's label moves when pressed.

Components combine four shadow tokens derived from these two values. A raised surface combines `--bevel` for the rims with `--raised` for the shadow beneath it. A well combines `--bevel-in` for the inverted rims with `--sunken` for the inner shadow. Because the shadows are translucent black, they look the same inside a colour context as outside one.

Because a custom property is computed on the element that declares it, the rims must be derived on the element whose face they describe. Inherited rims would match the parent's face instead. The `elevation` rule therefore derives them on every element and on `::before` and `::after`, which draw the marks inside checkboxes and switches. `--face` defaults to `--n-2`, the ladder's rung for raised surfaces. A component drawn with pseudo-elements of its own, such as a slider's track and thumb, calls `relief` to declare the same rims and shadows on them.

An application or a region therefore needs to set only `--relief` and `--face`. Setting `--relief: 0` on a region flattens everything inside it. Setting a different `--face` on one component changes that component's rims to match. To replace one component's relief outright, set `box-shadow` on that component.

`--floating` is the only shadow token that `--relief` doesn't scale. It is the drop shadow under an overlay, such as a dialog or a menu. It stays at full strength in a flat theme, because an overlay must still look as if it floats above the page.

In [ ]:
_relief = r"""
  --hi: oklch(from var(--face) calc(l + .16) c h / var(--relief));
  --lo: oklch(from var(--face) calc(l - .16) c h / var(--relief));
  --_relief-shadow: light-dark(rgb(0 0 0 / calc(.12 * var(--relief))), rgb(0 0 0 / calc(.5 * var(--relief))));
  --bevel: inset 1px 1px 0 var(--hi), inset -1px -1px 0 var(--lo);
  --bevel-in: inset 1px 1px 0 var(--lo), inset -1px -1px 0 var(--hi);
  --raised: 0 2px 4px var(--_relief-shadow);
  --sunken: inset 0 2px 4px var(--_relief-shadow);
"""
def relief(sel):
    "A CSS rule that derives rims and shadows from `--face` and `--relief` and declares them on `sel`"
    return f'{sel} {{{_relief}}}\n'

reg.rule('elevation', r"""
:root {
  --face: var(--n-2); --relief: 1;
  --_shadow-color: light-dark(rgb(0 0 0 / .12), rgb(0 0 0 / .5));
  --floating: 0 4px 12px var(--_shadow-color), 0 12px 32px var(--_shadow-color);
}
""" + relief('*, ::before, ::after'),
    layer='tokens', doc='Face-derived rims and raised and sunken shadows scaled by relief, with independent floating shadows.')

elevation [tokens]: Face-derived rims and raised and sunken shadows scaled by relief, with independent floating shadows.

In [ ]:
demo(swatches('box-shadow', ['bevel', 'raised', 'bevel-in', 'sunken', 'floating']))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;box-shadow:var(--bevel)">bevel</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;box-shadow:var(--raised)">raised</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;box-shadow:var(--bevel-in)">bevel-in</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;box-shadow:var(--sunken)">sunken</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;box-shadow:var(--floating)">floating</div>
</div>

```

## Customization

Register application overrides with `reg.rule` in its default layer, `app`, without editing the library's rules. Set the base values on `:root` for application-wide changes, or on a region for local ones. Colour is the exception: for a local hue, declare a colour context with `reg.context`, because the ladders are declared only on `:root` and on each context, not derived on every element. Here we choose a green hue, larger text, tighter spacing, and squarer corners:

In [ ]:
reg.rule('app-theme', r"""
:root { --hue: 145; --text: 1.125rem; --space: .875rem; --radius: .25rem; }
""", doc='Application-wide colour, type size, spacing, and roundness.')

app-theme [app]: Application-wide colour, type size, spacing, and roundness.

The same swatches now show the theme:

In [ ]:
demo(swatches('background', rungs), swatches('font-size', ['text-1', 'text', 'text1', 'text2', 'text3']),
    swatches('padding', ['space-3', 'space-2', 'space-1', 'space', 'space1', 'space2', 'space3']),
    swatches('border-radius', ['radius-1', 'radius', 'radius1', 'radius-full']))

```html
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-3)">n-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-2)">n-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-1)">n-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n)">n</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n1)">n1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n2)">n2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-3)">v-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-2)">v-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-1)">v-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v)">v</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v1)">v1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v2)">v2</div>
</div>
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text-1)">text-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text)">text</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text1)">text1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text2)">text2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;font-size:var(--text3)">text3</div>
</div>
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-3)">space-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-2)">space-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-1)">space-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space)">space</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space1)">space1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space2)">space2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space3)">space3</div>
</div>
<div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius-1)">radius-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius)">radius</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius1)">radius1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;border-radius:var(--radius-full)">radius-full</div>
</div>

```

Here two panels, each a region with a scale of its own, differ only in `--space` and `--text`. Each sets `font-size: var(--text)` for its own text, as described under Typography:


In [ ]:
def panel(title, style=''):
    "A heading, a line of text, and tiles showing three spacing steps, styled from the tokens"
    return Div(Div(title, style='font-size:var(--text2);font-weight:var(--weight1)'), Div('Body text at the region\'s size.'),
        swatches('padding', ['space-2', 'space', 'space1']), style=f'font-size:var(--text);padding:var(--space);display:grid;gap:var(--space-2);{style}')

demo(Div(panel('Normal'), panel('Compact', style='--space:.75rem;--text:.875rem'), style='display:grid;grid-template-columns:1fr 1fr'))

```html
<div style="display:grid;grid-template-columns:1fr 1fr">
  <div style="font-size:var(--text);padding:var(--space);display:grid;gap:var(--space-2);">
    <div style="font-size:var(--text2);font-weight:var(--weight1)">Normal</div>
    <div>Body text at the region's size.</div>
    <div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-2)">space-2</div>
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space)">space</div>
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space1)">space1</div>
    </div>
  </div>
  <div style="font-size:var(--text);padding:var(--space);display:grid;gap:var(--space-2);--space:.75rem;--text:.875rem">
    <div style="font-size:var(--text2);font-weight:var(--weight1)">Compact</div>
    <div>Body text at the region's size.</div>
    <div style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space-2)">space-2</div>
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space)">space</div>
      <div style="padding:.5rem;background:var(--n-2);font:12px monospace;padding:var(--space1)">space1</div>
    </div>
  </div>
</div>

```

A new colour context takes one `reg.context` call. Put `project=True` on a FastHTML element to give it and its contents this purple palette:


In [ ]:
reg.context('[project]', 300)


In [ ]:
demo(swatches('background', rungs, project=True))

```html
<div project style="display:flex;gap:.5rem;align-items:center;padding:1rem;background:var(--n-3);color:var(--n2)">
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-3)">n-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-2)">n-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n-1)">n-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n)">n</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n1)">n1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--n2)">n2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-3)">v-3</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-2)">v-2</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v-1)">v-1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v)">v</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v1)">v1</div>
  <div style="padding:.5rem;background:var(--n-2);font:12px monospace;background:var(--v2)">v2</div>
</div>

```

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()